# Capstone: Feature Engineering for the Netflix Greenlight Hackathon

This notebook is a companion to the capstone project instructions in Canvas
(*The Netflix Writers Dataset Pipeline*). Where that module 
explains **how** `netflix.titles.composite.csv`
gets built, this notebook is about what you do **after** it exists: turning
its raw columns into the kind of predictive factors a model — or a
screenwriter — can actually use.

Every sample below is a small, generic function. None of them are tied to
one specific column name more than necessary, so you can reuse the same
pattern throughout the project. Each one is followed by a short explanation of
*why* that factor matters when you're trying to predict whether a new TV
series concept has hit potential.

The composite dataset has one row per matched Netflix title, with these
columns:

| column | meaning |
|---|---|
| `netflix_title` | the title as it appears in Netflix's Top 10 data |
| `netflix_viewing_hours` | total hours viewed while the title was on the Top 10 list |
| `netflix_weeks` | number of weeks the title spent on the Top 10 list |
| `netflix_year_hint` | the year the title first appeared on the list |
| `tmdb_title`, `tmdb_popularity`, `tmdb_vote_average`, `tmdb_vote_count` | matched TMDB metadata |
| `imdb_title`, `imdb_averageRating`, `imdb_numVotes` | matched IMDb metadata |

A row can have missing `tmdb_*` or `imdb_*` values — the fuzzy-matching
pipeline from Module 2 doesn't find a confident match for every title, so
real-world coverage is never 100%.


## 1. Load the composite dataset

In your own environment this is just:

```python
df = pd.read_csv("netflix/data/netflix.titles.composite.csv")
```

Building the real file requires the full `make run` pipeline (Kaggle + IMDb
downloads), which takes 5–15 minutes and needs credentials this sandbox
doesn't have. So every sample below runs against a small **synthetic**
stand-in with the same columns and the same kinds of quirks — missing
matches, skewed vote counts, a rating or two based on a handful of votes —
so the code is genuinely exercised, not just described. Swap in your real
file and everything else works unchanged.


In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)
n = 300

titles = [f"Show {i:03d}" for i in range(n)]
netflix_weeks = rng.integers(1, 14, size=n)
netflix_viewing_hours = (netflix_weeks * rng.uniform(2_000_000, 9_000_000, size=n)).round(0)
netflix_year_hint = rng.integers(2021, 2026, size=n)

# Not every title gets a confident TMDB / IMDb match -- mirror that here.
has_tmdb = rng.random(n) > 0.15
has_imdb = rng.random(n) > 0.10

df = pd.DataFrame({
    "netflix_title": titles,
    "netflix_viewing_hours": netflix_viewing_hours,
    "netflix_weeks": netflix_weeks,
    "netflix_year_hint": netflix_year_hint,
    "tmdb_title": np.where(has_tmdb, titles, None),
    "tmdb_popularity": np.where(has_tmdb, rng.uniform(5, 400, size=n).round(1), np.nan),
    "tmdb_vote_average": np.where(has_tmdb, rng.uniform(4.5, 9.0, size=n).round(1), np.nan),
    "tmdb_vote_count": np.where(has_tmdb, rng.integers(20, 12_000, size=n), np.nan),
    "imdb_title": np.where(has_imdb, titles, None),
    "imdb_averageRating": np.where(has_imdb, rng.uniform(4.0, 9.3, size=n).round(1), np.nan),
    "imdb_numVotes": np.where(has_imdb, rng.integers(15, 300_000, size=n), np.nan),
})

df.shape, df.head(3)


((300, 11),
   netflix_title  netflix_viewing_hours  netflix_weeks  netflix_year_hint  \
 0      Show 000             48840931.0             13               2021   
 1      Show 001             68780466.0              9               2023   
 2      Show 002             60617686.0              9               2025   
 
   tmdb_title  tmdb_popularity  tmdb_vote_average  tmdb_vote_count imdb_title  \
 0   Show 000             38.1                7.4           9181.0   Show 000   
 1   Show 001            286.2                7.8           7116.0   Show 001   
 2   Show 002            120.8                5.0           5253.0   Show 002   
 
    imdb_averageRating  imdb_numVotes  
 0                 6.6       131664.0  
 1                 4.8        82446.0  
 2                 8.0        98769.0  )

## 2. Check match coverage before you engineer anything

Every downstream feature that depends on `tmdb_*` or `imdb_*` columns
inherits whatever coverage the fuzzy-matching step achieved. Check that
*first* — if only 40% of titles matched IMDb, any feature built on IMDb
ratings only describes 40% of your training data, and a model trained on it
will quietly generalize worse for the rest.

This is also directly relevant to the hackathon: Module 2's Section 9 surfaces
an actual bug in `fetch_imdb.py` where `REGEXP_REPLACE` runs before `LOWER` in
the `title_key` expression, silently stripping capital letters and lowering
match rates (`"Stranger Things"` → `"tranger hings"`). Fixing that bug
directly raises the number in the cell below — always worth checking after
a pipeline change.


In [2]:
def match_coverage(frame: pd.DataFrame, source_columns: dict[str, str]) -> pd.DataFrame:
    """Report what fraction of rows have a non-null match for each source.

    Args:
        frame: the composite dataset.
        source_columns: maps a human-readable source name to one column
            from that source to check for nulls, e.g. {"tmdb": "tmdb_title"}.

    Returns:
        A small DataFrame with one row per source: matched count, total
        count, and coverage as a percentage.
    """
    rows = []
    total = len(frame)
    for source_name, column in source_columns.items():
        matched = frame[column].notna().sum()
        rows.append({"source": source_name, "matched": matched, "total": total,
                      "coverage_pct": round(100 * matched / total, 1)})
    return pd.DataFrame(rows)


match_coverage(df, {"tmdb": "tmdb_title", "imdb": "imdb_title"})


,source,matched,total,coverage_pct
0,tmdb,243,300,81.0
1,imdb,266,300,88.7


## 3. Define the target: what counts as a "hit"?

Before engineering predictors, decide what they're predicting. A simple,
generic approach: call a title a hit if it lands in the top quantile of
some performance column — here, `netflix_viewing_hours`. This gives you a
binary label for classification, while keeping the raw column around for
regression if you'd rather predict a number than a category.


In [3]:
def label_hits(frame: pd.DataFrame, column: str, quantile: float = 0.75) -> pd.DataFrame:
    """Add an `is_hit` column: True if `column` is at or above the given quantile.

    Generic on purpose -- works on viewing hours today, or on any other
    numeric performance column you swap in later.
    """
    frame = frame.copy()
    threshold = frame[column].quantile(quantile)
    frame["is_hit"] = frame[column] >= threshold
    return frame


df = label_hits(df, "netflix_viewing_hours", quantile=0.75)
df["is_hit"].value_counts()


is_hit
False    225
True      75
Name: count, dtype: int64

## 4. Binge velocity: hours viewed per week on the list

Two shows can rack up the same total viewing hours in very different ways:
one gets devoured in two weeks, another lingers modestly for ten. Dividing
total hours by weeks-on-list surfaces that difference as a single number —
a rough proxy for how quickly word-of-mouth spreads. For a writer, this is
a hint that pacing and "bingeability" (short arcs, strong episode-ending
hooks) may matter as much as overall quality.


In [4]:
def binge_velocity(frame: pd.DataFrame,
                    hours_col: str = "netflix_viewing_hours",
                    weeks_col: str = "netflix_weeks") -> pd.Series:
    """Average viewing hours per week on the Top 10 list.

    Guards against division by zero for any title with weeks_col == 0.
    """
    weeks = frame[weeks_col].replace(0, np.nan)
    return (frame[hours_col] / weeks).rename("binge_velocity")


df["binge_velocity"] = binge_velocity(df)
df[["netflix_title", "netflix_viewing_hours", "netflix_weeks", "binge_velocity"]].head(3)


,netflix_title,netflix_viewing_hours,netflix_weeks,binge_velocity
0,Show 000,48840931.0,13,3.756995e+06
1,Show 001,68780466.0,9,7.642274e+06
2,Show 002,60617686.0,9,6.735298e+06


## 5. A trustworthy rating: shrink small samples toward the mean

A 9.4 rating from 12 IMDb votes and a 7.8 rating from 200,000 votes are not
equally informative, but a plain average treats them the same. This is the
same problem IMDb's own historical "weighted rating" formula addressed:
blend each title's rating with a prior (e.g. the dataset-wide mean),
weighted by how many votes back it up. A title with few votes gets pulled
toward the average; a title with many votes keeps its own score. This gives
a much more defensible "quality" feature than the raw rating column.


In [5]:
def bayesian_rating(frame: pd.DataFrame,
                     rating_col: str,
                     votes_col: str,
                     prior_votes: int = 500) -> pd.Series:
    """Shrink each rating toward the dataset mean based on vote count.

    weighted_rating = (v / (v + m)) * R + (m / (v + m)) * C

    where R is the title's own rating, v its vote count, C the dataset-wide
    mean rating, and m the `prior_votes` "confidence" threshold: titles with
    v << m get pulled hard toward C, titles with v >> m barely move.
    """
    v = frame[votes_col]
    R = frame[rating_col]
    C = frame[rating_col].mean()
    m = prior_votes
    return ((v / (v + m)) * R + (m / (v + m)) * C).rename(f"{rating_col}_shrunk")


df["imdb_rating_shrunk"] = bayesian_rating(df, "imdb_averageRating", "imdb_numVotes")
df[["netflix_title", "imdb_averageRating", "imdb_numVotes", "imdb_rating_shrunk"]].dropna().head(3)


,netflix_title,imdb_averageRating,imdb_numVotes,imdb_rating_shrunk
0,Show 000,6.6,131664.0,6.600282
1,Show 001,4.8,82446.0,4.811299
2,Show 002,8.0,98769.0,7.993323


## 6. Critic–audience alignment gap

TMDB's `vote_average` and IMDb's `averageRating` are both 0–10 scales
sourced from different, overlapping audiences. A small gap between them
suggests broad agreement across audience segments; a large gap suggests a
title (or script) that plays very differently to different crowds — divisive,
niche, or polarizing. Either can be a hit, but they're different bets, and
the gap itself is a genuinely new signal that neither source has alone.


In [6]:
def audience_alignment_gap(frame: pd.DataFrame,
                            col_a: str = "tmdb_vote_average",
                            col_b: str = "imdb_averageRating") -> pd.Series:
    """Absolute difference between two rating sources on the same scale."""
    return (frame[col_a] - frame[col_b]).abs().rename("audience_alignment_gap")


df["audience_alignment_gap"] = audience_alignment_gap(df)
df[["netflix_title", "tmdb_vote_average", "imdb_averageRating", "audience_alignment_gap"]].dropna().head(3)


,netflix_title,tmdb_vote_average,imdb_averageRating,audience_alignment_gap
0,Show 000,7.4,6.6,0.8
1,Show 001,7.8,4.8,3.0
2,Show 002,5.0,8.0,3.0


## 7. Buzz volume: tame skewed vote counts with a log transform

Vote counts (`tmdb_vote_count`, `imdb_numVotes`) span several orders of
magnitude — a handful of votes up to hundreds of thousands. Fed raw into a
linear model, a few enormous values dominate everything else. `log1p`
(log(1 + x), safe at zero) compresses that range into something a model can
actually use as a smooth "how much attention did this get" signal, distinct
from *how well* it was received.


In [7]:
def log_buzz(frame: pd.DataFrame, votes_col: str) -> pd.Series:
    """Log1p-transformed vote count -- a compressed 'attention volume' feature."""
    return np.log1p(frame[votes_col]).rename(f"{votes_col}_log")


df["imdb_buzz_log"] = log_buzz(df, "imdb_numVotes")
df[["netflix_title", "imdb_numVotes", "imdb_buzz_log"]].dropna().head(3)


,netflix_title,imdb_numVotes,imdb_buzz_log
0,Show 000,131664.0,11.788016
1,Show 001,82446.0,11.319911
2,Show 002,98769.0,11.500549


## 8. Assemble everything into one feature matrix

Wrap the samples above into a single pipeline function. This is the shape
you actually hand to a model: run it once, get back a clean numeric frame
plus the target column, ready for `SimpleLinearModel.fit()` from Module 0
or any scikit-learn estimator.


In [8]:
def build_feature_matrix(raw: pd.DataFrame) -> pd.DataFrame:
    """Turn the raw composite dataset into a model-ready feature matrix.

    Chains every feature function above. Rows with no IMDb match keep
    NaN for IMDb-derived features -- decide downstream whether to drop,
    impute, or model that missingness explicitly.
    """
    frame = label_hits(raw, "netflix_viewing_hours", quantile=0.75)
    frame["binge_velocity"] = binge_velocity(frame)
    frame["imdb_rating_shrunk"] = bayesian_rating(frame, "imdb_averageRating", "imdb_numVotes")
    frame["audience_alignment_gap"] = audience_alignment_gap(frame)
    frame["imdb_buzz_log"] = log_buzz(frame, "imdb_numVotes")

    feature_columns = [
        "binge_velocity",
        "imdb_rating_shrunk",
        "audience_alignment_gap",
        "imdb_buzz_log",
        "tmdb_popularity",
    ]
    return frame[["netflix_title", "is_hit", *feature_columns]]


features = build_feature_matrix(df)
features.describe()


,binge_velocity,imdb_rating_shrunk,audience_alignment_gap,imdb_buzz_log,tmdb_popularity
count,3.000000e+02,266.000000,216.000000,266.000000,243.000000
mean,5.542163e+06,6.673530,1.741204,11.649581,204.333333
std,2.045585e+06,1.480197,1.170695,1.027040,112.786259
min,2.108027e+06,4.014917,0.000000,6.459904,5.300000
25%,3.546625e+06,5.403707,0.700000,11.279534,115.150000
50%,5.761116e+06,6.699815,1.600000,11.976276,202.000000
75%,7.323639e+06,7.897830,2.525000,12.361748,295.250000
max,8.977398e+06,9.279912,4.700000,12.608537,399.800000


## 9. Sanity check: does each factor actually relate to the target?

Before trusting a feature, look at its correlation with `is_hit`. This
won't catch every useful nonlinear relationship, but it's a fast, generic
first filter — a feature with near-zero correlation and no clear domain
justification is a good candidate to drop or rethink.


In [9]:
numeric = features.drop(columns=["netflix_title"]).astype(float)
numeric.corr()["is_hit"].drop("is_hit").sort_values(key=abs, ascending=False)


binge_velocity            0.513524
tmdb_popularity           0.100953
audience_alignment_gap    0.022144
imdb_buzz_log             0.013298
imdb_rating_shrunk        0.005193
Name: is_hit, dtype: float64

## Where this goes next

These factors are inputs, not the model itself — Module 0's
`SimpleLinearModel` (or scikit-learn's `LogisticRegression`, if you're
predicting `is_hit` rather than raw hours) is what actually learns weights
for them. What this notebook gives you is the discipline of turning a messy,
partially-matched composite dataset into a small set of well-justified,
reusable numeric features, each one traceable to a specific hypothesis
about *why* a Netflix TV series succeeds — which is exactly the kind of
reasoning worth carrying back into your own script's premise.
